# WO7 — Climate classes: the class-relative inverse

Work order: `docs/cdop/pilot/wo7_climate-classes.md`. Findings: `docs/cdop/pilot/wo7_findings.md`.

Every similarity instrument so far is *query-relative* ("here is a place, paint its kin"). This builds the *class-relative* inverse: two independent discrete axes, computed and painted separately, with named cells at their cross-product.

- **Modality** {arid / aseasonal / 1-season / 2-season / undetermined} — arid gate (`pre_total < 100`) → cv gate (`cv < 0.20`) → Knoben ΔE (1 vs 2 seasons; undetermined when both sine models fit badly).
- **Phase** {warm-wet / cool-wet / weak coupling / no thermal cycle} — thermal gate (`tmp_seas_amp < 5 °C`) → sign & strength of the direct precip×temp correlation (WO6b Cell 19). Positive = rain with warmth (monsoon); negative = rain against warmth (Mediterranean).
- **Climate cell** = modality × phase. Named cells: Mediterranean = `cool-wet × 1-season`; monsoon = `warm-wet × 1-season`; twin-rains = `2-season`.

**Reused verbatim from WO6b:** Knoben ΔE (Cell 12) and the precip×temp correlation (Cell 19). Knoben is here **vectorized** (grid search over (δ, s), no per-basin scipy) and validated against the exact-Knoben verdicts before use — corpus-wide Nelder-Mead is infeasible.

**Accept gate (WO7 Part D):** `cool-wet × 1-season` paints the five classical Mediterranean regions and little else; `2-season` reproduces Knoben's published ~7% footprint (East Africa, Colombia, Sri Lanka, Indonesia).

In [ ]:
# Cell 1 — Setup
%matplotlib inline
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import geopandas as gpd
import pyogrio
from IPython.display import Image as IPImage
from scipy.optimize import minimize   # exact Knoben, for the validation cell only

from scripts.shared import db_utils

conn = db_utils.db_connect()
ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'cdop'
OUT.mkdir(parents=True, exist_ok=True)

WORLD = gpd.read_file(Path(pyogrio.__file__).parent / 'tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp')

MONTH_LETTERS = list('JFMAMJJASOND')
_t = np.arange(12)

# ── Declared conventions (all four stated in the legend as conventions, not discovered cuts) ──
THRESH_ARID   = 100.0   # mm/yr. The one threshold in a genuine histogram trough (wo2 Cell 12).
CV_FLAT       = 0.20    # aseasonal gate; WO6b Cell 18 plateau, separates Tennessee 0.144 / George Town 0.296.
THERMAL_FLOOR = 5.0     # deg C tmp_seas_amp; below this the temp curve is noise (WO6c Cell 7).
PT_CUT        = 0.50    # |precip x temp corr| cut for warm-wet / cool-wet vs weak coupling (WO6b Cell 19).
# Knoben verdict thresholds (WO6b Cell 12, verbatim):
POOR_E        = 0.25    # both tau fits above this => neither sine model describes the regime.
NEGLIGIBLE_D  = 0.02    # |dE| below this with poor fits => UNDETERMINED (method abstains).

print("Setup complete. Conventions:",
      f"ARID<{THRESH_ARID:.0f}mm  CV_FLAT<{CV_FLAT}  THERMAL<{THERMAL_FLOOR}C  |pt_corr|>={PT_CUT}")

In [ ]:
# Cell 2 — Corpus load, L06 (same pattern as WO6b Cell 2)
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

arr = pd.read_sql(
    "SELECT hybas_id, pre_mm_monthly, tmp_dc_monthly FROM public.v_basin06_persist_rev2", conn)
sc = pd.read_sql(
    "SELECT hybas_id, pre_mm_syr, tmp_dc_syr, "
    "ST_Y(ST_PointOnSurface(geom)) AS lat, ST_X(ST_PointOnSurface(geom)) AS lon "
    "FROM public.basin06", conn)
arr['hybas_id'] = arr['hybas_id'].astype(np.int64)
sc['hybas_id']  = sc['hybas_id'].astype(np.int64)
df = arr.merge(sc, on='hybas_id', how='left')

PRE_all = np.array(df['pre_mm_monthly'].tolist(), dtype=float)
TMP_all = np.array(df['tmp_dc_monthly'].tolist(), dtype=float)   # already deg C (persist view)

for col, scale in (('pre_mm_syr', 1.0), ('tmp_dc_syr', 0.1)):
    df[col] = df[col].astype(float)
    df.loc[df[col] == -9999 * scale, col] = np.nan
    df[col] = df[col] * scale

df['pre_total']    = PRE_all.sum(axis=1)
df['tmp_seas_amp'] = TMP_all.max(axis=1) - TMP_all.min(axis=1)

valid = (~np.isnan(PRE_all).any(axis=1) & ~np.isnan(TMP_all).any(axis=1)
         & (df['pre_total'] > 0) & df['tmp_dc_syr'].notna()).to_numpy()

df  = df[valid].reset_index(drop=True)
PRE = PRE_all[valid]
TMP = TMP_all[valid]
N   = len(df)

msg = [f"L06 valid: {N} of {len(valid)} basins",
       f"  arid (< {THRESH_ARID:.0f} mm/yr):        {(df['pre_total'] < THRESH_ARID).sum():6d}"
       f"  ({100*(df['pre_total'] < THRESH_ARID).mean():.1f}%)",
       f"  below thermal floor (<{THERMAL_FLOOR:.0f}C amp): {(df['tmp_seas_amp'] < THERMAL_FLOOR).sum():6d}"
       f"  ({100*(df['tmp_seas_amp'] < THERMAL_FLOOR).mean():.1f}%)"]
print("\n".join(msg))

In [ ]:
# Cell 3 — Probes (WO6b's set) resolved to row indices in df
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

PROBE_POINTS = [
    ('Mombasa',      -4.0435,   39.6682),
    ('Augsburg',     48.3705,   10.8978),
    ('Tbilisi',      41.6938,   44.8015),
    ('Kaifeng',      34.7986,  114.3413),
    ('Timbuktu',     16.8167,   -2.9833),
    ('George Town',   5.4141,  100.3288),
    ('Santiago',    -33.4489,  -70.6693),
    ('Yakutsk',      62.0280,  129.7326),
    ('Nairobi',      -1.2864,   36.8172),
]
PROBE_IDS = [('Somalia', 1060006860), ('Tennessee', 7060610850)]

# Exact-Knoben verdicts from WO6b Part C (the reference the vectorized method must reproduce).
WO6B_KNOBEN = {
    'Mombasa': 'BIMODAL', 'Augsburg': 'unimodal', 'Tbilisi': 'unimodal',
    'Kaifeng': 'unimodal', 'Timbuktu': 'unimodal', 'George Town': 'BIMODAL',
    'Santiago': 'unimodal', 'Yakutsk': 'unimodal', 'Nairobi': 'BIMODAL',
    'Somalia': 'BIMODAL', 'Tennessee': 'ASEASONAL',
}

_id_to_row = {int(h): i for i, h in enumerate(df['hybas_id'].to_numpy())}
def resolve_point(lat, lon):
    r = pd.read_sql(
        "SELECT hybas_id FROM basin06 "
        f"WHERE ST_Within(ST_SetSRID(ST_MakePoint({lon}, {lat}), 4326), geom) "
        "ORDER BY ST_Area(geom::geography) ASC LIMIT 1", conn)
    return int(r['hybas_id'].iloc[0]) if len(r) else None

probe_rows = {}          # name -> row index in df
for name, lat, lon in PROBE_POINTS:
    hid = resolve_point(lat, lon)
    if hid in _id_to_row:
        probe_rows[name] = _id_to_row[hid]
for name, hid in PROBE_IDS:
    if int(hid) in _id_to_row:
        probe_rows[name] = _id_to_row[int(hid)]

out = ["probe                idx    pre_total  tmp_amp   arid?"]
for name in WO6B_KNOBEN:
    if name not in probe_rows:
        out.append(f"  {name:14s} NOT RESOLVED"); continue
    i = probe_rows[name]
    out.append(f"  {name:14s} {i:7d}  {df['pre_total'][i]:8.0f}  {df['tmp_seas_amp'][i]:6.1f}"
               f"   {'ARID' if df['pre_total'][i] < THRESH_ARID else ''}")
print("\n".join(out))

In [ ]:
# Cell 4 — Knoben ΔE: exact (WO6b Cell 12) vs vectorized grid, validated before use.
# The class map needs Knoben on all 16k+ basins; per-basin Nelder-Mead is infeasible. This cell
# defines a vectorized (delta, s) grid search and proves it reproduces exact-Knoben verdicts on
# the WO6b synthetics AND the 11 probes. Only the grid version is used downstream.
# ONE print for the whole cell.

def _cr(delta):
    """B&W eq 4 truncation-correction factor; zero below delta=1. Works on scalar or array."""
    d = np.asarray(delta, dtype=float)
    poly = -0.001*d**4 + 0.026*d**3 - 0.245*d**2 + 0.2432*d - 0.038
    return np.where(d > 1, poly, 0.0)

def _verdict(e12, e6, cv):
    """Shared three-way+ verdict (identical thresholds for exact and grid)."""
    if not np.isfinite(cv):        return 'UNDETERMINED'
    if cv < CV_FLAT:               return 'ASEASONAL'
    dE = e12 - e6
    if min(e12, e6) > POOR_E and abs(dE) < NEGLIGIBLE_D:
        return 'UNDETERMINED'
    return 'BIMODAL' if dE > 0 else 'unimodal'

# ── exact (scipy), from WO6b Cell 12 verbatim ────────────────────────────────────────────
def _knoben_sim(pbar, delta, s, tau):
    return np.maximum(0.0, pbar * (1.0 + float(_cr(delta)) + delta * np.sin(2*np.pi*(_t - s)/tau)))
def _fit_exact(p_obs, tau):
    pbar = p_obs.mean()
    if pbar <= 0: return np.nan
    def obj(par):
        delta, s = par
        return 1e6 if delta < 0 else np.abs(_knoben_sim(pbar, delta, s, tau) - p_obs).mean()/pbar
    best = np.inf
    for s0 in np.linspace(0, tau, 6, endpoint=False):
        for d0 in (0.3, 0.8, 1.5, 3.0):
            r = minimize(obj, x0=[d0, s0], method='Nelder-Mead',
                         options={'xatol':1e-6,'fatol':1e-8,'maxiter':2000})
            best = min(best, float(r.fun))
    return best
def knoben_verdict_exact(p):
    m = p.mean(); cv = float(p.std()/m) if m > 0 else np.nan
    return _verdict(_fit_exact(p, 12.0), _fit_exact(p, 6.0), cv)

# ── vectorized grid ──────────────────────────────────────────────────────────────────────
def knoben_E_grid(P, tau, d_step=0.05, s_per_month=4):
    """Best Knoben E per row of P (M,12) over a (delta, s) grid. Returns E (M,)."""
    P = np.atleast_2d(P).astype(float)
    pbar = P.mean(axis=1)
    ok = pbar > 0
    grid_delta = np.arange(0.0, 4.0 + 1e-9, d_step)
    grid_s = np.linspace(0.0, tau, int(round(tau*s_per_month)), endpoint=False)
    sin_tab = np.sin(2*np.pi*(_t[None, :] - grid_s[:, None])/tau)   # (n_s, 12)
    bestE = np.full(P.shape[0], np.inf)
    safe_pbar = np.where(ok, pbar, 1.0)
    for d in grid_delta:
        cr = float(_cr(d))
        for row in sin_tab:
            sim = np.maximum(0.0, safe_pbar[:, None]*(1.0 + cr + d*row[None, :]))
            obj = np.abs(sim - P).mean(axis=1)/safe_pbar
            bestE = np.minimum(bestE, obj)
    bestE[~ok] = np.nan
    return bestE

def knoben_verdict_grid_rows(P, cv):
    e12 = knoben_E_grid(P, 12.0); e6 = knoben_E_grid(P, 6.0)
    return np.array([_verdict(e12[i], e6[i], cv[i]) for i in range(len(P))]), e12, e6

# ── synthetic validation (known modality by construction, WO6b Cell 12) ───────────────────
def _mk(peaks, base=10.0, width=1.2):
    v = np.full(12, float(base))
    for m_, h in peaks:
        d = np.minimum(np.abs(_t - m_), 12 - np.abs(_t - m_))
        v = v + h * np.exp(-0.5*(d/width)**2)
    return v
cases = {
    'unimodal sharp (monsoon)':   _mk([(7, 200)], base=2),
    'unimodal broad':             _mk([(7, 60)],  base=40, width=2.2),
    'bimodal symmetric':          _mk([(3, 100), (9, 100)], base=20),
    'bimodal asym 1.5:1':         _mk([(3, 150), (9, 100)], base=20),
    'bimodal asym 2:1':           _mk([(3, 200), (9, 100)], base=20),
    'bimodal asym 4:1':           _mk([(3, 200), (9, 50)],  base=20),
    'aseasonal flat':             _mk([], base=80),
    'aseasonal + noise':          _mk([(2, 12), (6, 9), (10, 11)], base=100, width=1.0),
    'weakly seasonal (Tenn.)':    _mk([(2, 20), (11, 18)], base=100, width=2.0),
}
Psyn = np.array(list(cases.values()))
cv_syn = Psyn.std(axis=1)/Psyn.mean(axis=1)
vg_syn, e12s, e6s = knoben_verdict_grid_rows(Psyn, cv_syn)

out = ["=== synthetic validation: exact vs grid (must agree) ===",
       f"  {'case':26s} {'exact':13s} {'grid':13s} {'match':5s}"]
n_ok = 0
for (label, v), vg in zip(cases.items(), vg_syn):
    ve = knoben_verdict_exact(v)
    ok = (ve == vg); n_ok += ok
    out.append(f"  {label:26s} {ve:13s} {vg:13s} {'OK' if ok else 'XXX'}")
out.append(f"  synthetics agree: {n_ok}/{len(cases)}")

# ── probe validation: grid vs exact vs WO6b reference ─────────────────────────────────────
pnames = [n for n in WO6B_KNOBEN if n in probe_rows]
Pprobe = PRE[[probe_rows[n] for n in pnames]]
cv_probe = Pprobe.std(axis=1)/Pprobe.mean(axis=1)
vg_probe, e12p, e6p = knoben_verdict_grid_rows(Pprobe, cv_probe)
out += ["", "=== probe validation: grid vs exact vs WO6b Cell-12 reference ===",
        f"  {'probe':13s} {'exact':13s} {'grid':13s} {'WO6b':11s} {'ok':4s}"]
n_ok = 0
for name, vg in zip(pnames, vg_probe):
    ve = knoben_verdict_exact(PRE[probe_rows[name]])
    ref = WO6B_KNOBEN[name]
    ok = (vg == ve == ref); n_ok += ok
    out.append(f"  {name:13s} {ve:13s} {vg:13s} {ref:11s} {'OK' if ok else 'XXX'}")
out.append(f"  probes agree (grid==exact==WO6b): {n_ok}/{len(pnames)}")
print("\n".join(out))

In [ ]:
# Cell 5 — Modality axis on the full L06 corpus (arid gate -> cv gate -> Knoben ΔE)
# Knoben runs only on the modality-eligible subset (non-arid, cv>=CV_FLAT); the two gates
# resolve the rest with no fit. Times the corpus run to settle where this computation can live.
import time

cv_all = PRE.std(axis=1) / np.where(PRE.mean(axis=1) > 0, PRE.mean(axis=1), np.nan)
df['cv'] = cv_all

modality = np.full(N, 'undetermined', dtype=object)
arid_mask      = df['pre_total'].to_numpy() < THRESH_ARID
aseasonal_mask = (~arid_mask) & np.isfinite(cv_all) & (cv_all < CV_FLAT)
eligible       = (~arid_mask) & (~aseasonal_mask) & np.isfinite(cv_all)

modality[arid_mask]      = 'arid'
modality[aseasonal_mask] = 'aseasonal'

t0 = time.time()
e12 = knoben_E_grid(PRE[eligible], 12.0)
e6  = knoben_E_grid(PRE[eligible], 6.0)
dt  = time.time() - t0
dE  = e12 - e6

sub = np.full(int(eligible.sum()), '1-season', dtype='<U12')
sub[dE > 0] = '2-season'
undet = (np.minimum(e12, e6) > POOR_E) & (np.abs(dE) < NEGLIGIBLE_D)
sub[undet] = 'undetermined'
modality[np.where(eligible)[0]] = sub
df['modality'] = modality

MOD_ORDER = ['arid', 'aseasonal', '1-season', '2-season', 'undetermined']
vc = pd.Series(modality).value_counts()
out = [f"Knoben grid on {int(eligible.sum())} eligible basins: {dt:.1f}s "
       f"(implies L08 ~190k -> ~{dt*190675/max(int(eligible.sum()),1):.0f}s)",
       "", "modality shares (L06):"]
for k in MOD_ORDER:
    c = int(vc.get(k, 0))
    out.append(f"  {k:14s} {c:6d}  ({100*c/N:5.1f}%)")
out += ["", "probe modality classes:",
        f"  {'probe':13s} {'pre_tot':>7} {'cv':>6} {'modality':13s} {'(WO6b Knoben)':13s}"]
for name in WO6B_KNOBEN:
    if name not in probe_rows: continue
    i = probe_rows[name]
    out.append(f"  {name:13s} {df['pre_total'][i]:7.0f} {cv_all[i]:6.2f} "
               f"{modality[i]:13s} {WO6B_KNOBEN[name]:13s}")
print("\n".join(out))

In [ ]:
# Cell 6 — Phase axis: direct precip x temp correlation + thermal gate (WO6b Cell 19)
# One dot product per basin, no sine fit (so defined for bimodal basins too). The thermal gate
# separates "no thermal cycle" (temp curve is noise, <5C amp) from genuine "weak coupling".
# ONE print.

def _pearson_rows(A, B):
    Ac = A - A.mean(axis=1, keepdims=True)
    Bc = B - B.mean(axis=1, keepdims=True)
    num = (Ac * Bc).sum(axis=1)
    den = np.sqrt((Ac**2).sum(axis=1) * (Bc**2).sum(axis=1))
    return np.where(den > 0, num/den, np.nan)

pt_corr = _pearson_rows(PRE, TMP)
df['pt_corr'] = pt_corr

no_thermal = df['tmp_seas_amp'].to_numpy() < THERMAL_FLOOR
phase = np.full(N, 'weak coupling', dtype=object)
phase[pt_corr >=  PT_CUT] = 'warm-wet'
phase[pt_corr <= -PT_CUT] = 'cool-wet'
phase[no_thermal]         = 'no thermal cycle'   # gate wins: overrides any correlation label
df['phase'] = phase

PHASE_ORDER = ['warm-wet', 'cool-wet', 'weak coupling', 'no thermal cycle']

# Shares WITHOUT the thermal gate (raw pt_corr cut) vs WITH it — the WO's question.
fin = np.isfinite(pt_corr)
raw_ww = 100*(pt_corr[fin] >=  PT_CUT).mean()
raw_cw = 100*(pt_corr[fin] <= -PT_CUT).mean()
raw_wk = 100 - raw_ww - raw_cw
vc = pd.Series(phase).value_counts()

out = ["corpus pt_corr distribution:", pd.Series(pt_corr[fin]).describe().to_string(), "",
       f"WITHOUT thermal gate (|pt_corr| cut at {PT_CUT}, over {int(fin.sum())} basins):",
       f"  warm-wet   {raw_ww:5.1f}%     cool-wet {raw_cw:5.1f}%     weak {raw_wk:5.1f}%",
       f"  (WO6b Cell 19 reported ~55 / ~17 / ~28 at this cut, no gate)", "",
       f"WITH thermal gate (<{THERMAL_FLOOR:.0f}C amp -> 'no thermal cycle'):"]
for k in PHASE_ORDER:
    c = int(vc.get(k, 0))
    out.append(f"  {k:18s} {c:6d}  ({100*c/N:5.1f}%)")
n_gated = int(no_thermal.sum())
out.append(f"  -> the gate reclassified {n_gated} basins ({100*n_gated/N:.1f}%) that the raw cut "
           f"would have split into warm/cool/weak")
out += ["", "probe phase classes:",
        f"  {'probe':13s} {'tmp_amp':>7} {'pt_corr':>8} {'phase':18s}"]
for name in WO6B_KNOBEN:
    if name not in probe_rows: continue
    i = probe_rows[name]
    out.append(f"  {name:13s} {df['tmp_seas_amp'][i]:7.1f} {pt_corr[i]:8.3f} {phase[i]:18s}")
print("\n".join(out))

In [ ]:
# Cell 7 — Climate cell = modality × phase cross-product (Option A). Named cells fall out here.
# ONE print (crosstab via to_string).

df['cell'] = df['modality'].astype(str) + ' | ' + df['phase'].astype(str)

# Named cells (the human-legible labels; everything else keeps its "modality | phase" id).
def name_cell(mod, ph):
    if mod == '2-season':                              return 'twin-rains'
    if mod == '1-season' and ph == 'cool-wet':         return 'Mediterranean'
    if mod == '1-season' and ph == 'warm-wet':         return 'monsoon / summer-rain'
    if mod == 'arid':                                  return 'arid'
    if mod == 'aseasonal':                             return 'aseasonal'
    return None
df['cell_named'] = [name_cell(m, p) for m, p in zip(df['modality'], df['phase'])]

ct = pd.crosstab(df['modality'], df['phase'])
ct = ct.reindex(index=[m for m in MOD_ORDER if m in ct.index],
                columns=[p for p in PHASE_ORDER if p in ct.columns], fill_value=0)
ct_pct = (100*ct/N).round(1)

out = ["modality × phase counts (L06):", ct.to_string(), "",
       "as % of corpus:", ct_pct.to_string(), "",
       "named cells (the Part D targets):"]
named_counts = pd.Series([c for c in df['cell_named'] if c]).value_counts()
for nm in ['Mediterranean', 'monsoon / summer-rain', 'twin-rains', 'arid', 'aseasonal']:
    c = int(named_counts.get(nm, 0))
    out.append(f"  {nm:24s} {c:6d}  ({100*c/N:5.1f}%)")
out += ["", "probe cells:",
        f"  {'probe':13s} {'modality':13s} {'phase':18s} {'named':22s}"]
for name in WO6B_KNOBEN:
    if name not in probe_rows: continue
    i = probe_rows[name]
    out.append(f"  {name:13s} {df['modality'][i]:13s} {df['phase'][i]:18s} "
               f"{str(df['cell_named'][i]):22s}")
print("\n".join(out))

In [ ]:
# Cell 8 — Global maps of the two axes, painted as filled basin polygons (a real choropleth).
# Rep-point scatter was too sparse/pale to read; this fills every basin, matching the pmtiles
# product. Geometry loads ONCE (cached in GDF); re-running the map cells is then fast.
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
import matplotlib.patches as mpatches

if 'GDF' not in globals():
    print("loading basin06 geometries (simplified, one-time)...")
    _g = gpd.read_postgis(
        "SELECT hybas_id, ST_SimplifyPreserveTopology(geom, 0.05) AS geom FROM public.basin06",
        conn, geom_col='geom').rename_geometry('geometry')
    _g['hybas_id'] = _g['hybas_id'].astype(np.int64)
    GDF = _g.merge(df[['hybas_id', 'modality', 'phase', 'cell', 'cell_named']],
                   on='hybas_id', how='inner')

# Darkened so the light classes read on white.
MOD_COLORS = {'arid':'#D9B879', 'aseasonal':'#8A8A8A', '1-season':'#2E8B57',
              '2-season':'#8E44AD', 'undetermined':'#C879B0'}
PHASE_COLORS = {'warm-wet':'#D7301F', 'cool-wet':'#2166AC',
                'weak coupling':'#9E9E9E', 'no thermal cycle':'#E8C34A'}

def plot_classes(ax, series, order, cmap, title):
    ax.set_facecolor('white')
    GDF.plot(ax=ax, color=series.map(cmap).fillna('#eeeeee').tolist(),
             linewidth=0, edgecolor='none')
    WORLD.boundary.plot(ax=ax, color='#777777', linewidth=0.3)
    ax.set_title(title, color='black', fontsize=12)
    ax.set_xlim(-180, 180); ax.set_ylim(-60, 85)
    ax.set_xticks([]); ax.set_yticks([])
    counts = series.value_counts()
    handles = [mpatches.Patch(color=cmap[k], label=f"{k} ({int(counts.get(k,0))})") for k in order]
    leg = ax.legend(handles=handles, loc='lower left', fontsize=8, framealpha=0.92)
    for txt in leg.get_texts(): txt.set_color('black')

print("drawing modality + phase axis maps...")
fig, axes = plt.subplots(2, 1, figsize=(15, 13))
fig.patch.set_facecolor('white')
plot_classes(axes[0], GDF['modality'], MOD_ORDER, MOD_COLORS,
             "Modality axis (L06)  —  arid gate → cv gate → Knoben ΔE")
plot_classes(axes[1], GDF['phase'], PHASE_ORDER, PHASE_COLORS,
             "Phase axis (L06)  —  precip×temp correlation + 5°C thermal gate")
fig.tight_layout()
outp = OUT / 'wo7_axes_L06.png'
fig.savefig(outp, dpi=140, facecolor='white', bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outp)))

In [ ]:
# Cell 9 — Part D acid test: paint each named cell (members filled, rest light grey).
# Mediterranean = cool-wet × 1-season ; summer-rain = warm-wet × 1-season (broad, NOT "monsoon" —
# it swallows Yakutsk/Augsburg, Cell 7) ; twin-rains = 2-season.

def highlight_cell(ax, mask, color, title):
    ax.set_facecolor('white')
    colr = np.where(mask.to_numpy(), color, '#ececec')
    GDF.plot(ax=ax, color=colr, linewidth=0, edgecolor='none')
    WORLD.boundary.plot(ax=ax, color='#999999', linewidth=0.3)
    n = int(mask.sum())
    ax.set_title(f"{title}   (n={n}, {100*n/len(GDF):.1f}%)", color='black', fontsize=12)
    ax.set_xlim(-180, 180); ax.set_ylim(-60, 85)
    ax.set_xticks([]); ax.set_yticks([])

g_med  = GDF['cell_named'] == 'Mediterranean'
g_summ = GDF['cell_named'] == 'monsoon / summer-rain'
g_twin = GDF['modality'] == '2-season'

print("drawing Part D named-cell maps...")
fig, axes = plt.subplots(3, 1, figsize=(15, 19))
fig.patch.set_facecolor('white')
highlight_cell(axes[0], g_med,  '#2166AC', "Mediterranean  =  cool-wet × 1-season")
highlight_cell(axes[1], g_summ, '#D7301F', "Summer-rain (thermally coupled)  =  warm-wet × 1-season")
highlight_cell(axes[2], g_twin, '#8E44AD', "Twin-rains  =  2-season   (Knoben Fig 1c target ~7%)")
fig.tight_layout()
outp = OUT / 'wo7_named_cells_L06.png'
fig.savefig(outp, dpi=140, facecolor='white', bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outp)))

In [ ]:
# Cell 10 — Part D quantitative gate + arid-margin speckle check + save classification.
# "and little else" = fraction of a named cell falling OUTSIDE its published target regions.
# Self-contained in df-space (not dependent on the map cells). ONE print; writes the parquet.

lat = df['lat'].to_numpy(); lon = df['lon'].to_numpy()
geo_ok = np.isfinite(lat) & np.isfinite(lon)
med_mask  = (df['cell_named'] == 'Mediterranean').to_numpy()
twin_mask = (df['modality'] == '2-season').to_numpy()

def in_boxes(la, lo, boxes):
    m = np.zeros(len(la), bool)
    for (lo1, lo2, la1, la2) in boxes:
        m |= (lo >= lo1) & (lo <= lo2) & (la >= la1) & (la <= la2)
    return m

# Five classical Mediterranean-climate regions (generous bounding boxes).
MED_BOXES = [(-10, 42, 30, 46),    # Mediterranean basin + Levant
             (-124, -116, 31, 43), # California
             (-73, -69, -38, -28), # central Chile
             (16, 27, -35, -30),   # Cape
             (114, 120, -36, -29), # SW Australia
             (135, 141, -38, -32)] # SE/S Australia (Adelaide)
# Knoben twin-rains concentrations.
TWIN_BOXES = [(32, 52, -6, 13),    # East Africa / Horn
              (-80, -69, -3, 12),  # Colombia / N Andes
              (78, 82, 5, 10),     # Sri Lanka
              (95, 142, -11, 9)]   # Maritime SE Asia / Indonesia

med_in  = med_mask  & geo_ok & in_boxes(lat, lon, MED_BOXES)
twin_in = twin_mask & geo_ok & in_boxes(lat, lon, TWIN_BOXES)
nmed, ntwin = int((med_mask & geo_ok).sum()), int((twin_mask & geo_ok).sum())

out = ["=== Part D quantitative gate ===", "",
       f"Mediterranean cell (cool-wet × 1-season): n={nmed}",
       f"  inside the 5 classical regions:  {int(med_in.sum()):5d}  ({100*med_in.sum()/max(nmed,1):.1f}%)",
       f"  leak (outside all 5):            {nmed-int(med_in.sum()):5d}  "
       f"({100*(nmed-med_in.sum())/max(nmed,1):.1f}%)   <- 'and little else' test", "",
       f"Twin-rains cell (2-season): n={ntwin}",
       f"  inside Knoben's regions:         {int(twin_in.sum()):5d}  ({100*twin_in.sum()/max(ntwin,1):.1f}%)",
       f"  elsewhere:                       {ntwin-int(twin_in.sum()):5d}  "
       f"({100*(ntwin-twin_in.sum())/max(ntwin,1):.1f}%)", ""]

# Arid-margin speckle: 2-season basins just above the arid gate (100-300 mm) that fall OUTSIDE
# the twin-rains regions are the candidate spurious-bimodal calls (WO7 note to self).
margin = twin_mask & geo_ok & (df['pre_total'].to_numpy() >= THRESH_ARID) & (df['pre_total'].to_numpy() < 300)
margin_out = margin & ~in_boxes(lat, lon, TWIN_BOXES)
out += ["=== arid-margin speckle check (2-season, 100-300 mm/yr) ===",
        f"  in 100-300 mm band:              {int(margin.sum()):5d}",
        f"  of those, OUTSIDE twin regions:  {int(margin_out.sum()):5d}  "
        f"(candidate spurious bimodal — should be small)", ""]

# Save for the extraction step (categorical route + similarity lens read this).
save = df[['hybas_id', 'modality', 'phase', 'cell', 'cell_named',
           'pre_total', 'cv', 'tmp_seas_amp', 'pt_corr', 'lat', 'lon']].copy()
sp = OUT / 'wo7_climate_classes_L06.parquet'
save.to_parquet(sp, index=False)
out.append(f"saved {len(save)} rows -> {sp}")
print("\n".join(out))

In [ ]:
# Cell 11 — Diagnostic 1 (numbers): twin-rains at L08. Does Indonesia return (L06 island-aggregation),
# and does the mid-latitude (Central Asia) band shrink? L08 arrays cached in globals for re-runs.
# NUMBERS ONLY — the map is Cell 11b (a figure in the same cell would overwrite this text in PyCharm).
import warnings; warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
import time

if 'PRE8' not in globals():
    print("loading L08 corpus (one-time)...")
    a8 = pd.read_sql("SELECT hybas_id, pre_mm_monthly, tmp_dc_monthly "
                     "FROM public.v_basin08_persist_rev2", conn)
    s8 = pd.read_sql("SELECT hybas_id, ST_Y(ST_PointOnSurface(geom)) AS lat, "
                     "ST_X(ST_PointOnSurface(geom)) AS lon FROM public.basin08", conn)
    a8['hybas_id'] = a8['hybas_id'].astype(np.int64); s8['hybas_id'] = s8['hybas_id'].astype(np.int64)
    d8 = a8.merge(s8, on='hybas_id', how='left')
    PRE8_all = np.array(d8['pre_mm_monthly'].tolist(), float)
    TMP8_all = np.array(d8['tmp_dc_monthly'].tolist(), float)
    pt8 = PRE8_all.sum(axis=1)
    v8 = np.isfinite(PRE8_all).all(1) & np.isfinite(TMP8_all).all(1) & (pt8 > 0)
    D8 = d8[v8].reset_index(drop=True)
    PRE8 = PRE8_all[v8]; TMP8 = TMP8_all[v8]; PT8 = pt8[v8]

N8  = len(D8)
cv8 = PRE8.std(1) / np.where(PRE8.mean(1) > 0, PRE8.mean(1), np.nan)
mod8 = np.full(N8, 'undetermined', dtype=object)
arid8 = PT8 < THRESH_ARID
asea8 = (~arid8) & np.isfinite(cv8) & (cv8 < CV_FLAT)
elig8 = (~arid8) & (~asea8) & np.isfinite(cv8)
mod8[arid8] = 'arid'; mod8[asea8] = 'aseasonal'
t0 = time.time()
E12 = knoben_E_grid(PRE8[elig8], 12.0); E6 = knoben_E_grid(PRE8[elig8], 6.0)
dt = time.time() - t0
dEE = E12 - E6
s8v = np.full(int(elig8.sum()), '1-season', dtype='<U12')
s8v[dEE > 0] = '2-season'
s8v[(np.minimum(E12, E6) > POOR_E) & (np.abs(dEE) < NEGLIGIBLE_D)] = 'undetermined'
mod8[np.where(elig8)[0]] = s8v

lat8 = D8['lat'].to_numpy(); lon8 = D8['lon'].to_numpy(); geo8 = np.isfinite(lat8) & np.isfinite(lon8)
twin8 = (mod8 == '2-season')
twin6 = (df['modality'] == '2-season').to_numpy()

INDO_BOX = [(95, 141, -10, 8)]
i6 = twin6 & geo_ok & in_boxes(lat, lon, INDO_BOX)
i8 = twin8 & geo8   & in_boxes(lat8, lon8, INDO_BOX)
k8 = twin8 & geo8   & in_boxes(lat8, lon8, TWIN_BOXES)
m8 = twin8 & geo8 & (PT8 >= THRESH_ARID) & (PT8 < 300)
m8o = m8 & ~in_boxes(lat8, lon8, TWIN_BOXES)

out = [f"L08 modality (Knoben grid on {int(elig8.sum())} eligible in {dt:.1f}s):",
       f"  2-season (twin-rains): {int(twin8.sum())}  ({100*twin8.sum()/N8:.1f}%)   "
       f"[L06 was 899, 5.5%]", "",
       "Indonesia box (95-141E, -10..8N) twin-rains count:",
       f"  L06: {int(i6.sum())}     L08: {int(i8.sum())}   <- does the maritime continent return?", "",
       f"L08 twin-rains inside Knoben regions: {int(k8.sum())}/{int(twin8.sum())} "
       f"({100*k8.sum()/max(int(twin8.sum()),1):.1f}%)   [L06 was 25.6%]",
       f"L08 arid-margin speckle (100-300mm, outside twin regions): {int(m8o.sum())} "
       f"({100*m8o.sum()/max(int(twin8.sum()),1):.1f}% of twin)   [L06 was 196, 21.8%]"]
print("\n".join(out))

In [ ]:
# Cell 11b — Diagnostic 1 (map): L08 twin-rains footprint. Reuses globals from Cell 11.
# Separate cell so it does NOT overwrite Cell 11's numbers in the PyCharm console.
print("drawing L08 twin-rains footprint...")
fig, ax = plt.subplots(figsize=(15, 7))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')
WORLD.plot(ax=ax, color='#efefef', edgecolor='#cccccc', linewidth=0.3)
ax.scatter(lon8[twin8 & geo8], lat8[twin8 & geo8], s=2.5, c='#8E44AD',
           marker='.', linewidths=0, rasterized=True)
ax.set_title(f"Twin-rains at L08  (2-season, n={int(twin8.sum())})  —  vs L06 for the Indonesia check",
             color='black', fontsize=12)
ax.set_xlim(-180, 180); ax.set_ylim(-60, 85); ax.set_xticks([]); ax.set_yticks([])
outp = OUT / 'wo7_twin_L08.png'
fig.savefig(outp, dpi=140, facecolor='white', bbox_inches='tight')
plt.close(fig)
display(IPImage(str(outp)))

In [ ]:
# Cell 12 — Diagnostic 2: characterize the Mediterranean-cell leak (the 841 outside the 5 regions).
# Is the leak cold-winter-continental (Iran/Central Asia), separable by a mild-winter floor? If so,
# quantify how a coldest-month-temp floor trades leak-cut against in-region retention. ONE print.

coldest = TMP.min(axis=1)                 # coldest-month mean temp, deg C (per-basin)
tann    = df['tmp_dc_syr'].to_numpy()     # annual mean temp
in_reg = med_mask & geo_ok & in_boxes(lat, lon, MED_BOXES)
leak   = med_mask & geo_ok & ~in_boxes(lat, lon, MED_BOXES)

def _desc(mask, v):
    x = v[mask]
    return f"n={int(mask.sum()):4d}  coldest-mo: med {np.median(x):5.1f}  " \
           f"[{np.percentile(x,10):5.1f}, {np.percentile(x,90):5.1f}]"

out = ["=== Mediterranean cell: in-region vs leak, coldest-month temp ===",
       f"  in-region (5 classic): {_desc(in_reg, coldest)}",
       f"  leak (outside):        {_desc(leak, coldest)}",
       "  (classic Med has mild winters; cold-continental winter-rain has cold ones)", "",
       "=== coldest-month floor sweep: leak cut vs in-region retained ===",
       f"  {'floor(coldest>)':16s} {'leak kept':>10} {'leak cut%':>10} {'in-reg kept':>12} {'in-reg cut%':>12}"]
nleak, nreg = int(leak.sum()), int(in_reg.sum())
for thr in (-5, -2, 0, 3, 5, 8):
    lk = leak & (coldest > thr)
    rg = in_reg & (coldest > thr)
    out.append(f"  {('%.0f'%thr):>13s}    {int(lk.sum()):>10} {100*(nleak-lk.sum())/max(nleak,1):>9.0f}% "
               f"{int(rg.sum()):>12} {100*(nreg-rg.sum())/max(nreg,1):>11.0f}%")
out += ["",
        "Read: a floor that cuts most of the leak while retaining most of the 5 classic regions is",
        "the price of Option B (Mediterranean = cool-wet x 1-season x mild-winter). If leak and",
        "in-region overlap heavily in coldest-month temp, no clean floor exists -> Option A (rename)."]
print("\n".join(out))

In [15]:
# Cell 13 — Diagnostic 3 (aridity, per WO7a register note): the third dial Diagnostic 2 didn't test.
# What excludes the Iran/Central-Asia winter-rain belt from Koppen-Med is ARIDITY, not cold winters
# (Iranian summers are hot). So test an annual-total floor: does "keep only wetter basins" cleanly
# drop the leak while retaining the 5 classic regions? ONE print.

in_reg = med_mask & geo_ok & in_boxes(lat, lon, MED_BOXES)
leak   = med_mask & geo_ok & ~in_boxes(lat, lon, MED_BOXES)
ptot   = df['pre_total'].to_numpy()

def _descp(mask):
    x = ptot[mask]
    return f"n={int(mask.sum()):4d}  annual mm: med {np.median(x):5.0f}  " \
           f"[{np.percentile(x,10):5.0f}, {np.percentile(x,90):5.0f}]"

out = ["=== Mediterranean cell: in-region vs leak, ANNUAL PRECIP TOTAL ===",
       f"  in-region (5 classic): {_descp(in_reg)}",
       f"  leak (outside):        {_descp(leak)}",
       "  (Koppen Cs excludes the Iranian/C-Asian steppe by aridity; the dry leak should be lower)", "",
       "=== annual-total floor sweep: keep wetter basins (pre_total > X mm) ===",
       f"  {'floor(mm>)':10s} {'leak kept':>10} {'leak cut%':>10} {'in-reg kept':>12} {'in-reg cut%':>12}"]
nleak, nreg = int(leak.sum()), int(in_reg.sum())
for thr in (150, 250, 350, 450, 600):
    lk = leak & (ptot > thr); rg = in_reg & (ptot > thr)
    out.append(f"  {('%d'%thr):>7s}    {int(lk.sum()):>10} {100*(nleak-lk.sum())/max(nleak,1):>9.0f}% "
               f"{int(rg.sum()):>12} {100*(nreg-rg.sum())/max(nreg,1):>11.0f}%")

# The leak is not monolithic: a WET Pacific-NW portion (winter rain, high total) that no aridity
# floor can remove -- so even a perfect aridity dial addresses only the dry-continental half.
pnw = leak & (lon >= -126) & (lon <= -116) & (lat > 43)
pnw_med = np.median(ptot[pnw]) if int(pnw.sum()) else float('nan')
out += ["",
        f"Leak is not all dry: Pacific-NW-ish portion (lon -126..-116, lat>43): n={int(pnw.sum())}, "
        f"annual mm med {pnw_med:.0f} -- wet winter-rain an aridity floor cannot touch.",
        "",
        "Read: if leak and in-region overlap on annual total the way they did on winter temp (Cell 12),",
        "no clean aridity floor exists either -> Option A holds, now on BOTH candidate dials."]
print("\n".join(out))

=== Mediterranean cell: in-region vs leak, ANNUAL PRECIP TOTAL ===
  in-region (5 classic): n= 484  annual mm: med   468  [  175,   900]
  leak (outside):        n= 841  annual mm: med   251  [  125,  1254]
  (Koppen Cs excludes the Iranian/C-Asian steppe by aridity; the dry leak should be lower)

=== annual-total floor sweep: keep wetter basins (pre_total > X mm) ===
  floor(mm>)  leak kept  leak cut%  in-reg kept  in-reg cut%
      150           637        24%          444           8%
      250           421        50%          398          18%
      350           293        65%          323          33%
      450           234        72%          256          47%
      600           194        77%          159          67%

Leak is not all dry: Pacific-NW-ish portion (lon -126..-116, lat>43): n=75, annual mm med 757 -- wet winter-rain an aridity floor cannot touch.

Read: if leak and in-region overlap on annual total the way they did on winter temp (Cell 12),
no clean aridity floor